# 2. Pré-processamento de Dados

Este notebook é responsável pelo pipeline de transformação. Ele extrai os dados brutos da camada `raw`, utiliza a camada `interim` para processamento temporário (descompactação) e promove os datasets formatados no padrão YOLO para a camada final `processed`.

## Configuração e Importações

In [1]:
import json
import random
import shutil
import sys
import zipfile
from pathlib import Path
from typing import Dict, Any, List

try:
    from PIL import Image
except ImportError:
    raise ImportError("Por favor instale Pillow: uv add Pillow")

# Adiciona a raiz do projeto ao caminho para importação do módulo.
sys.path.append("..")
from src import config

print(f"Diretório raw: {config.RAW_DIR}")
print(f"Diretório interim: {config.INTERIM_DIR}")
print(f"Diretório processado: {config.PROCESSED_DIR}")

Diretório raw: /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/raw
Diretório interim: /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/interim
Diretório processado: /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/processed


## 2.1 Processamento: Roboflow Datasets

Os datasets oriundos do Roboflow já foram baixados de forma normalizada pelo pipeline interno da API. O objetivo aqui é apenas promovê-los da camada `raw` para a `processed`, criando o arquivo de validação de dependência.

In [2]:
def process_roboflow_datasets(raw_dir: Path, processed_dir: Path) -> None:
    f"""
    Promove os conjuntos de dados Roboflow normalizados diretamente do diretório raw
    para o diretório processed, tornando-os prontos para o treinamento YOLO.
    """
    roboflow_datasets: List[str] = [
        "inside-bus-view", 
        "passenger-deakin", 
        # "passenger-detection-bus"
    ]
    
    print("Iniciando a promoção de conjuntos de dados do Roboflow...")
    
    for dataset_name in roboflow_datasets:
        src_path: Path = raw_dir / dataset_name
        dst_path: Path = processed_dir / dataset_name
        
        if not src_path.exists():
            print(f"Dataset {dataset_name} não encontrado em {src_path}.")
            continue

        # Limpe o diretório de destino para evitar a mistura de dados.
        if dst_path.exists():
            shutil.rmtree(dst_path)
            
        # Copie a "árvore" do conjunto de dados, do formato bruto para o processado.
        shutil.copytree(src_path, dst_path)
        
        # Crie o marcador de validação oculto exigido pelo script de treinamento.
        marker_file: Path = dst_path / ".download_complete"
        marker_file.write_text("ok\n", encoding="utf-8")
        
        print(f"Processado e promovido com sucesso. {dataset_name}.")

# Execute a promoção para todos os conjuntos de dados do Roboflow.
process_roboflow_datasets(config.RAW_DIR, config.PROCESSED_DIR)

Iniciando a promoção de conjuntos de dados do Roboflow...
Processado e promovido com sucesso. inside-bus-view.
Processado e promovido com sucesso. passenger-deakin.


## 2.2 Processamento: CrowdHuman Dataset

O CrowdHuman possui uma estrutura complexa nativa. O pipeline extrairá as imagens dos arquivos `.zip` para a área de rascunho (`interim`), cruzará os dados com os metadados brutos (`.odgt`), extrairá as coordenadas da classe "person" convertendo-as para YOLO, e salvará a amostragem configurada em `processed`.

In [3]:
def process_crowdhuman_pipeline(
    raw_dir: Path,
    interim_dir: Path,
    processed_dir: Path,
    n_train: int = 800,
    n_val: int = 200,
    n_test: int = 200
) -> None:
    f"""
    Executa todo o pipeline de dados do CrowdHuman:
    1. Extrai arquivos ZIP brutos do diretório raw para o diretório provisório.
    2. Analisa as anotações ODGT e normaliza as caixas delimitadoras para o formato YOLO.
    3. Salva as divisões finais e balanceadas do conjunto de dados no diretório processado.
    """
    raw_ch_dir: Path = raw_dir / "CrowdHuman"
    interim_ch_dir: Path = interim_dir / "CrowdHuman_Extracted"
    processed_ch_dir: Path = processed_dir / "crowdhuman"
    
    if not raw_ch_dir.exists():
        print(f"Diretório bruto do CrowdHuman não encontrado em {raw_ch_dir}. Execute primeiro o Notebook 1 de Download.")
        return
        
    print("1. Extraindo arquivos ZIP brutos para um diretório intermediário...")
    images_ext_dir: Path = interim_ch_dir / "Images"
    images_ext_dir.mkdir(parents=True, exist_ok=True)
    
    zip_files: List[Path] = list(raw_ch_dir.glob("*.zip"))
    for z_path in zip_files:
        print(f"   -> Extracting {z_path.name}...")
        with zipfile.ZipFile(z_path, 'r') as zip_ref:
            zip_ref.extractall(images_ext_dir)
            
    print("2. Analisando anotações ODGT...")
    def parse_odgt(odgt_path: Path) -> List[Dict[str, Any]]:
        f"""Lê um arquivo ODGT e retorna uma lista de dicionários."""
        records: List[Dict[str, Any]] = []
        with open(odgt_path, 'r', encoding='utf-8') as f_in:
            for line in f_in:
                records.append(json.loads(line.strip()))
        return records
        
    train_odgt: Path = raw_ch_dir / "annotation_train.odgt"
    val_odgt: Path = raw_ch_dir / "annotation_val.odgt"
    
    train_records: List[Dict[str, Any]] = parse_odgt(train_odgt) if train_odgt.exists() else []
    val_records: List[Dict[str, Any]] = parse_odgt(val_odgt) if val_odgt.exists() else []
    
    image_paths_map: Dict[str, Path] = {p.name: p for p in images_ext_dir.rglob("*.jpg")}
    
    train_records = [r for r in train_records if f"{r['ID']}.jpg" in image_paths_map]
    val_records = [r for r in val_records if f"{r['ID']}.jpg" in image_paths_map]
    
    random.seed(42)
    random.shuffle(train_records)
    random.shuffle(val_records)
    
    selected_test: List[Dict[str, Any]] = train_records[:n_test]
    selected_train: List[Dict[str, Any]] = train_records[n_test : n_test + n_train]
    selected_val: List[Dict[str, Any]] = val_records[:n_val]
    
    splits: Dict[str, List[Dict[str, Any]]] = {
        "train": selected_train,
        "valid": selected_val,
        "test": selected_test
    }
    
    if processed_ch_dir.exists():
        shutil.rmtree(processed_ch_dir)
        
    print("3. Convertendo para o formato YOLO e salvando no diretório de arquivos processados...")
    for split_name, records in splits.items():
        print(f"   -> Processando {split_name} ({len(records)} images)...")
        img_dir: Path = processed_ch_dir / split_name / "images"
        lbl_dir: Path = processed_ch_dir / split_name / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        
        for rec in records:
            img_id: str = rec["ID"]
            img_filename: str = f"{img_id}.jpg"
            src_img: Path = image_paths_map[img_filename]
            dst_img: Path = img_dir / img_filename
            
            shutil.copy(src_img, dst_img)
            
            img_w: int
            img_h: int
            with Image.open(dst_img) as img:
                img_w, img_h = img.size
                
            yolo_boxes: List[str] = []
            if "gtboxes" in rec:
                for box_info in rec["gtboxes"]:
                    if box_info.get("tag") != "person":
                        continue
                        
                    bx: float
                    by: float
                    bw: float
                    bh: float
                    if "fbox" in box_info:
                        bx, by, bw, bh = box_info["fbox"]
                    elif "vbox" in box_info:
                        bx, by, bw, bh = box_info["vbox"]
                    else:
                        continue
                        
                    cx: float = (bx + bw / 2.0) / img_w
                    cy: float = (by + bh / 2.0) / img_h
                    nw: float = bw / img_w
                    nh: float = bh / img_h
                    
                    cx = max(0.0, min(1.0, cx))
                    cy = max(0.0, min(1.0, cy))
                    nw = max(0.0, min(1.0, nw))
                    nh = max(0.0, min(1.0, nh))
                    
                    yolo_boxes.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
                    
            dst_lbl: Path = lbl_dir / f"{img_id}.txt"
            dst_lbl.write_text("\n".join(yolo_boxes), encoding="utf-8")
            
    yaml_path: Path = processed_ch_dir / "data.yaml"
    yaml_content: str = (
        "train: ../train/images\n"
        "val: ../valid/images\n"
        "test: ../test/images\n"
        "nc: 1\n"
        "names: ['person']\n"
    )
    yaml_path.write_text(yaml_content, encoding="utf-8")
    
    (processed_ch_dir / ".download_complete").write_text("ok\n", encoding="utf-8")
    print(f"Processo concluído com sucesso. Conjunto de dados final salvo em {processed_ch_dir}")

In [4]:
# Execute o pipeline com as restrições de conjunto de dados desejadas.
process_crowdhuman_pipeline(
    raw_dir=config.RAW_DIR,
    interim_dir=config.INTERIM_DIR,
    processed_dir=config.PROCESSED_DIR,
    n_train=800,
    n_val=200,
    n_test=200
)

1. Extraindo arquivos ZIP brutos para um diretório intermediário...
   -> Extracting CrowdHuman_val.zip...
   -> Extracting CrowdHuman_train01.zip...
2. Analisando anotações ODGT...
3. Convertendo para o formato YOLO e salvando no diretório de arquivos processados...
   -> Processando train (800 images)...
   -> Processando valid (200 images)...
   -> Processando test (200 images)...
Processo concluído com sucesso. Conjunto de dados final salvo em /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/processed/crowdhuman
